# SC203 Kaggle DoSSR smoke runner

Run this in Kaggle Notebooks with Internet enabled and a GPU accelerator (P100 or T4). Attach Kaggle Datasets containing `dossr_default.ckpt` and the extracted StableSR `RealSRVal_crop128` images. The checkpoint stays in `/kaggle/input` or can already exist in `/kaggle/working/weights`; the notebook does not download or copy it. All generated configs, logs, smoke images, and outputs go under `/kaggle/working`.

Push the selected branch from your Mac before running the Git cell. The offload code and Kaggle path handling are **UNTESTED** until a Kaggle run succeeds.

In [ ]:
# CELL 1 — Inspect Kaggle's runtime before installing dependencies.
import platform, sys, torch
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
# CELL 2 — Clone or update the chosen public Git branch. No token is used.
from pathlib import Path
import subprocess

BRANCH = 'feat/baseline-runner'  # Update this after pushing a different branch.
REPO_URL = 'https://github.com/lemon4life/SC203.git'
REPO_DIR = Path('/kaggle/working/SC203')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
WORK_DIR = Path('/kaggle/working')
LOG_DIR = WORK_DIR / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
print('Repo:', REPO_DIR, 'branch:', BRANCH)

In [ ]:
# CELL 3 — Kaggle preflight and dependency setup.
# Reports Kaggle's existing pip-check conflicts, then dry-runs DoSSR requirements while pinning torch/torchvision.
# Stops before installation if resolution fails or proposes changing torch/torchvision.
!{sys.executable} -u /kaggle/working/SC203/scripts/colab_setup.py --allow-preexisting-conflicts --requirements /kaggle/working/SC203/external/DOSSR/requirements.txt --log-dir /kaggle/working/logs

In [ ]:
# CELL 4 — Locate the attached checkpoint and RealSR folder, then prepare one-image config.
# Auto-discovery requires exactly one dossr_default.ckpt and one RealSRVal_crop128 folder under /kaggle/input.
# If discovery is ambiguous, add --checkpoint /kaggle/input/<dataset>/.../dossr_default.ckpt
# and/or --input-dir /kaggle/input/<dataset>/.../RealSRVal_crop128 to this command.
!{sys.executable} -u /kaggle/working/SC203/scripts/prepare_kaggle_smoke.py --assets-root /kaggle/input --smoke-dir /kaggle/working/realsr_smoke --config-out /kaggle/working/configs/baseline_kaggle_smoke.yaml --output-root /kaggle/working/outputs/baseline-kaggle-smoke --log-dir /kaggle/working/logs

In [ ]:
%%bash
# CELL 5 — Run one image with FP16 plus condition-encoder offload. Uses cuda:1 when visible; otherwise CPU fallback.
cd /kaggle/working/SC203
set -o pipefail
python -u src/run_dossr.py --config /kaggle/working/configs/baseline_kaggle_smoke.yaml 2>&1 | tee /kaggle/working/baseline_kaggle_smoke_console.log
status=${PIPESTATUS[0]}
echo "KAGGLE_SMOKE_EXIT_STATUS=$status"
exit "$status"

In [ ]:
# CELL 6 — Confirm exactly one output before considering a full run.
from pathlib import Path
result_dir = Path('/kaggle/working/outputs/baseline-kaggle-smoke/seed_42')
pngs = sorted(result_dir.glob('*.png')) if result_dir.is_dir() else []
print('Output folder:', result_dir)
print('PNG count:', len(pngs))
for path in pngs: print(path)
assert len(pngs) == 1, 'Smoke test did not produce exactly one PNG.'
print('Kaggle one-image smoke test passed (execution only; results remain UNTESTED scientifically).')